# Mock Interview Practice — Day 22
**FAANG / Top Product Company Level — Week 4, Day 1**

*Days 15–21 built ADVANCED/HARD product-company skills. Week 4 opens the FAANG tier: Google, Amazon, Meta, Uber, Microsoft, Netflix, Adobe, Flipkart. Questions now blend deep systems knowledge with 2025–2026 trending topics: streaming lakehouse tables, GenAI data pipelines, vector search, A/B experimentation, Declarative Automation Bundles, and zero-trust governance.*

**18 questions · 8 categories · 10 [HARD] + 8 [FAANG]**

| Q | Category | Topic | Tag |
| --- | --- | --- | --- |
| Q1–Q2 | SQL Advanced | OHLC candles + as-of quote enrichment · A/B experiment integrity (SRM + CUPED) | FAANG ×2 |
| Q3–Q4 | Python | Count-Min Sketch heavy hitters · text-to-SQL guardrails | FAANG ×2 |
| Q5–Q6 | PySpark | Streaming dedup state semantics · IVF vector index + ANN search | HARD / FAANG |
| Q7–Q8 | Spark Optimization | Shuffle mechanics + push-based shuffle · Lakehouse Federation pushdown | HARD ×2 |
| Q9–Q11 | ETL | GenAI doc ingestion (idempotent + DLQ + circuit breaker) · cross-cloud DR · Streaming Tables vs MVs | FAANG / HARD ×2 |
| Q12–Q13 | Data Modeling | RAG corpus as a governed data product · time-series metrics platform model | FAANG / HARD |
| Q14–Q15 | Snowflake | Zero-trust security stack · Snowpark Container Services | HARD ×2 |
| Q16–Q18 | Monitoring | DABs env promotion · real-time anomaly detection · PII access compliance | FAANG / HARD ×2 |

**Today's thread**: market-data SQL and experiment rigor (Q1–Q2) → probabilistic structures and GenAI safety (Q3–Q4) → streaming state and vector serving (Q5–Q6) → engine internals and federation (Q7–Q8) → ingestion failure semantics and DR (Q9–Q11) → data products for AI and observability (Q12–Q13) → security and platform ops (Q14–Q18).

**Rule**: attempt every question in your head or on paper BEFORE opening the solution cell. Score yourself honestly in the scorecard at the end.

## Q1 — SQL Advanced: Market-Data OHLC Candles + As-Of Quote Enrichment [FAANG]

**Interviewer (Robinhood):** Our tick store has irregularly-spaced trades and a quotes feed. Two deliverables on streaming-safe SQL:

1. **OHLC candlesticks**: build 5-minute bars (open / high / low / close) per symbol from raw ticks — `open` must be the price of the *earliest* trade in the bucket, `close` the *latest* (NOT min/max of price!).
2. **As-of quote enrichment**: for every trade, attach the *most recent quote at or before* the trade timestamp (the last bid/ask the exchange would have shown), and compute the effective spread. Equality joins fail here — you need point-in-time semantics.

**Requirements**
- Use a tumbling window on `window(ts, '5 minutes')` and the **struct trick** — `min(struct(ts, price)).price` — to get order-aware open/close in ONE aggregation.
- Show the as-of join TWO ways: (a) `pyspark.pandas.merge_asof` (direction backward), (b) pure SQL — range join + `row_number() OVER (PARTITION BY trade ORDER BY quote_ts DESC)`.
- Explain where a native `ASOF JOIN` exists (DBSQL warehouses), and why the range-join variant needs a `RANGE_JOIN` hint at scale (the naive join is O(trades × quotes-in-overlap)).

**What the interviewer wants**: bucketed time-series aggregation without pandas resampling, point-in-time join semantics (never leaking future quotes), and knowing the struct-ordering trick.

In [0]:
# Solution Q1 — OHLC candles + as-of quote enrichment
from pyspark.sql import functions as F

ticks = spark.createDataFrame(
    [("UBER","2026-11-03 09:31:05", 61.10),("UBER","2026-11-03 09:33:41", 61.45),
     ("UBER","2026-11-03 09:38:12", 61.30),("UBER","2026-11-03 09:42:59", 61.70),
     ("UBER","2026-11-03 09:47:00", 61.55),("UBER","2026-11-03 09:59:31", 61.80),
     ("UBER","2026-11-03 10:03:02", 61.95)],
    "symbol string, ts string, price double").withColumn("ts", F.to_timestamp("ts"))
quotes = spark.createDataFrame(
    [("UBER","2026-11-03 09:30:00", 61.08, 61.12),("UBER","2026-11-03 09:37:30", 61.28, 61.32),
     ("UBER","2026-11-03 09:46:45", 61.52, 61.58),("UBER","2026-11-03 10:01:00", 61.90, 61.99)],
    "symbol string, ts string, bid double, ask double").withColumn("ts", F.to_timestamp("ts"))
ticks.createOrReplaceTempView("ticks"); quotes.createOrReplaceTempView("quotes")

# --- Part 1: 5-minute OHLC. struct(ts, price) sorts by ts first -> min = open, max = close
candles = spark.sql("""
  SELECT symbol,
         window(ts, '5 minutes').start AS bar_start,
         min(struct(ts, price)).price AS open_,
         max(price)  AS high,
         min(price)  AS low,
         max(struct(ts, price)).price AS close_
  FROM ticks GROUP BY symbol, window(ts, '5 minutes') ORDER BY bar_start
""")
candles.show(truncate=False)

# --- Part 2a: as-of via pyspark.pandas merge_asof (backward = last quote <= trade ts)
import pyspark.pandas as ps
t_ps = ps.from_pandas(ticks.toPandas()).sort_values("ts")
q_ps = ps.from_pandas(quotes.toPandas()).sort_values("ts")
merged = ps.merge_asof(t_ps, q_ps, on="ts", by="symbol", direction="backward")
merged = merged.assign(spread=lambda d: d["ask"] - d["bid"])
print(merged[["symbol","ts","price","bid","ask","spread"]].sort_values("ts").head(10).to_string())

# --- Part 2b: pure SQL as-of: range join + latest-quote-per-trade window
sql_asof = spark.sql("""
  WITH joined AS (
    SELECT t.symbol, t.ts AS trade_ts, t.price, q.ts AS quote_ts, q.bid, q.ask,
           row_number() OVER (PARTITION BY t.symbol, t.ts ORDER BY q.ts DESC) AS rn
    FROM ticks t JOIN quotes q
      ON t.symbol = q.symbol AND q.ts <= t.ts
  )
  SELECT symbol, trade_ts, price, quote_ts, bid, ask, round(ask - bid, 4) AS spread
  FROM joined WHERE rn = 1 ORDER BY trade_ts
""")
sql_asof.show(truncate=False)

import math
m = merged.to_pandas()
row = m[m["ts"].astype(str).str.startswith("2026-11-03 09:31")].iloc[0]
s = [r for r in sql_asof.collect() if str(r["trade_ts"]).startswith("2026-11-03 09:31")][0]
assert math.isclose(row["spread"], 61.12 - 61.08, abs_tol=1e-9)   # quote 09:30:00 backs trade 09:31:05
assert math.isclose(s["spread"], 0.04, abs_tol=1e-9)              # SQL variant agrees
print("Q1 OK: OHLC bars + two as-of implementations agree (spread=0.04 at 09:31:05)")
print()
print('Native ASOF JOIN is a Databricks SQL (warehouse) operator:')
print('  SELECT t.* FROM trades t ASOF LEFT JOIN quotes q')
print('    ON t.symbol = q.symbol MATCH_CONDITION t.ts >= q.ts')
print('On Spark compute, merge_asof or the range-join + row_number pattern above is the equivalent;')
print('at scale add /*+ RANGE_JOIN(q, 10) */ so the engine interval-joins instead of cross-joining overlap.')

## Q2 — SQL Advanced: A/B Experiment Integrity — SRM + CUPED [FAANG]

**Interviewer (Meta):** A dashboard claims a big revenue lift from the new checkout flow. You get one look at the assignment log before anyone celebrates.

**Requirements**
1. **Sample Ratio Mismatch (SRM)**: 20 users were assigned 15/5 control/treatment. Run a chi-square goodness-of-fit vs the expected 50/50 (critical value 3.841 at df=1, α=0.05) in ONE SQL pass. If it fires, the experiment is invalid — say why you must NOT read the lift.
2. **CUPED**: even with clean assignment, variance kills sensitivity. Given a pre-exposure covariate `pre` (revenue in the 4 weeks before exposure), compute θ = cov(post, pre)/var(pre) and the CUPED-adjusted variance `var(post) − θ²·var(pre)`. Prove the variance reduction equals corr(post, pre)².
3. Show the adjusted lift: `naive_lift − θ·(mean_pre_T − mean_pre_C)` and explain why it shrinks when pre-period means are imbalanced.

**What the interviewer wants**: gate-keeping instincts (SRM check BEFORE reading any lift), the CUPED identity, and why pre-period covariates are free sensitivity at Meta/Uber scale.

In [0]:
# Solution Q2 — SRM chi-square + CUPED variance reduction
from pyspark.sql import functions as F

exp_df = spark.createDataFrame(
    [("u01","control",4.2,3.9),("u02","control",3.8,4.0),("u03","control",4.6,4.4),
     ("u04","control",3.9,3.7),("u05","control",4.1,4.2),("u06","control",3.7,3.8),
     ("u07","control",4.3,4.1),("u08","control",4.0,4.3),("u09","control",4.4,4.0),
     ("u10","control",3.8,3.9),("u11","control",4.2,4.4),("u12","control",4.5,4.3),
     ("u13","control",4.0,4.1),("u14","control",4.3,4.2),("u15","control",3.9,3.8),
     ("u21","treatment",4.9,4.6),("u22","treatment",4.4,4.7),("u23","treatment",5.1,4.8),
     ("u24","treatment",4.7,4.5),("u25","treatment",4.3,4.6)],
    "user_id string, variant string, pre double, post double")
exp_df.createOrReplaceTempView("ab_exp")

result = spark.sql("""
  WITH pooled AS (
    SELECT count(*) AS n,
           var_samp(post) AS var_post, var_samp(pre) AS var_pre,
           covar_samp(post, pre) AS cov,
           avg(CASE WHEN variant='treatment' THEN post END) AS t_post,
           avg(CASE WHEN variant='control'   THEN post END) AS c_post,
           avg(CASE WHEN variant='treatment' THEN pre  END) AS t_pre,
           avg(CASE WHEN variant='control'   THEN pre  END) AS c_pre,
           sum(CASE WHEN variant='treatment' THEN 1 ELSE 0 END) AS n_t,
           sum(CASE WHEN variant='control'   THEN 1 ELSE 0 END) AS n_c
    FROM ab_exp
  )
  SELECT
    (n_t - n/2.0)*(n_t - n/2.0)/(n/2.0) + (n_c - n/2.0)*(n_c - n/2.0)/(n/2.0) AS srm_chi2,
    cov / var_pre AS theta,
    var_post AS var_naive,
    var_post - power(cov / var_pre, 2) * var_pre AS var_cuped,
    t_post - c_post AS naive_lift,
    (t_post - c_post) - (cov / var_pre) * (t_pre - c_pre) AS adjusted_lift
  FROM pooled
""").collect()[0]
chi2, theta = float(result["srm_chi2"]), float(result["theta"])
var_naive, var_cuped = float(result["var_naive"]), float(result["var_cuped"])
naive_lift, adj_lift = float(result["naive_lift"]), float(result["adjusted_lift"])
print(f"SRM chi2 = {chi2:.3f} (crit 3.841 @ df=1) -> {'SRM DETECTED - STOP, do not read the lift' if chi2 > 3.841 else 'no SRM'}")
print(f"theta = {theta:.4f}, var(post) {var_naive:.4f} -> CUPED var {var_cuped:.4f} (reduction {(1-var_cuped/var_naive)*100:.1f}% = corr^2)")
print(f"naive lift {naive_lift:.4f} -> adjusted lift {adj_lift:.4f} (pre-imbalance corrected)")

import pandas as pd, math
pdf = exp_df.toPandas()
corr2 = pdf["pre"].corr(pdf["post"])**2
assert chi2 > 3.841                                  # 15/5 in 20 users -> chi2 = 5.0: experiment invalid
assert abs((1 - var_cuped/var_naive) - corr2) < 1e-9  # CUPED identity: variance reduction == corr^2
assert math.isclose(theta, pdf["post"].cov(pdf["pre"]) / pdf["pre"].var(ddof=1), rel_tol=1e-9)
assert adj_lift < naive_lift                          # imbalanced pre-periods inflate the naive lift
print("Q2 OK: SRM gate fires; CUPED identity holds; adjusted lift de-biased")

## Q3 — Python: Count-Min Sketch — Heavy Hitters on an Unbounded Stream [FAANG]

**Interviewer (Twitter/X):** The trending-topics service sees millions of events per minute and must report every topic exceeding 1% of stream volume, with **O(1) memory per event** and no exact Counters dict (unbounded keys would blow RAM).

You already know Bloom filters (membership, Day 21) and HyperLogLog (cardinality, Days 10/14). This is the third leg of the sketch family: **frequency estimation**.

**Requirements**
1. Implement a Count-Min Sketch: `d` pairwise-independent hash rows × `w` counters, `add(item)` bumps every row's cell, `estimate(item)` = min across rows.
2. Add **conservative update**: never raise any cell above `min(current cells) + count` — prove why it only tightens the over-estimate.
3. Stream 20,000 events (3 heavy topics + 9,000 unique noise keys + 50 mid-tier keys), then:
 - assert every estimate is within the CM guarantee `estimate ≤ true + ε·N` with `ε = e/w`;
 - assert the >1% heavy-hitter cut returns EXACTLY {uber, lyft, taxi} — no false heavies from hash collisions;
 - report the memory saved vs an exact Counter (exact: 9,050 keys; sketch: `w×d` counters, ~KBs).

**What the interviewer wants**: which sketch answers which question (membership vs cardinality vs frequency), why over-estimation is one-sided and bounded, and the conservative-update trick.

In [0]:
# Solution Q3 — Count-Min Sketch with conservative update
import hashlib
from collections import Counter

class CountMinSketch:
    def __init__(self, width=512, depth=5, seed=42):
        self.w, self.d = width, depth
        self.table = [[0]*width for _ in range(depth)]
        self.seeds = [seed*1000 + i for i in range(depth)]
        self.total = 0

    def _positions(self, item):
        b = str(item).encode()
        for row, s in enumerate(self.seeds):
            h = int.from_bytes(hashlib.blake2b(b, digest_size=8, key=str(s).encode()).digest(), 'big')
            yield row, h % self.w

    def add(self, item, count=1):
        self.total += count
        cells = list(self._positions(item))
        cur = min(self.table[r][c] for r, c in cells)
        # conservative update: a cell already holds another key's over-estimate; only raise to cur+count
        for r, c in cells:
            self.table[r][c] = max(self.table[r][c], cur + count)

    def estimate(self, item):
        return min(self.table[r][c] for r, c in self._positions(item))

stream = (["uber"]*5000 + ["lyft"]*3000 + ["taxi"]*1000
          + [f"session_{i}" for i in range(9000)]      # unique noise keys
          + [f"evt_{i%50}" for i in range(2000)])       # mid-tier (40 each)
exact = Counter(stream)
cms = CountMinSketch()
for ev in stream:
    cms.add(ev)

w, d = cms.w, cms.d
eps = 2.718281828 / w                        # CM guarantee: estimate <= true + eps*N with prob 1-e^-d
guarantee = exact["uber"] + eps * cms.total
assert cms.estimate("uber") <= guarantee, (cms.estimate("uber"), guarantee)
assert all(cms.estimate(k) >= exact[k] for k in exact)   # one-sided error: NEVER under-estimates
hh = {k for k in exact if cms.estimate(k) >= 0.01 * cms.total}
assert hh == {"uber", "lyft", "taxi"}                     # exact heavy set, no collision false-positives
assert all(cms.estimate(k) < 0.01 * cms.total for k in exact if exact[k] <= 40)

print(f"stream = {cms.total:,} events | exact keys = {len(exact):,} | sketch memory = {w*d} counters (~{w*d*8/1024:.1f} KB)")
print(f"heavy hitters (>1% of stream): {sorted(hh, key=cms.estimate, reverse=True)}")
print(f"CM guarantee check: est(uber)={cms.estimate('uber')} <= true({exact['uber']}) + eps*N({eps*cms.total:.0f})")
print("Q3 OK: one-sided bounded error, exact heavy set, O(w*d) memory vs unbounded exact dict")
print()
print("Sketch cheat-sheet: Bloom=membership, HLL=cardinality, Count-Min=frequency/heavy hitters;")
print("conservative update keeps cell over-estimates as low as the smallest colliding cell, never worse.")

## Q4 — Python: Text-to-SQL Guardrails — Making an LLM's SQL Safe to Execute [FAANG]

**Interviewer (Databricks GenAI platform):** Our internal text-to-SQL agent returns SQL that analysts want to run on governed tables. Before execution, you own the **guardrail layer**. (Day 20 built the *evaluation harness* for answer quality — this question is the *safety* side, which is what production incidents are made of.)

**Requirements**
Build `guard(sql) -> (safe_sql, decisions)` enforcing:
1. **Read-only**: block INSERT/UPDATE/DELETE/MERGE/CREATE/DROP/ALTER/TRUNCATE/GRANT/REVOKE/COPY/EXPORT.
2. **Table allowlist**: every table referenced in FROM/JOIN must be in the approved set; unknown tables BLOCK.
3. **No SELECT \***: rewrite to an explicit column list (cost + governance + PII column control).
4. **Row/cost cap**: inject `LIMIT 1000` when missing; clamp it when larger.
5. Return a decision per issue: `BLOCK` (never execute) vs `FIX` (rewrite) — and a final execute/abort verdict.

**Test cases**: benign `SELECT *` (fix), oversized LIMIT (clamp), query against `main.ops.secret_payouts` (block), a `DELETE` (block), and a legit 2-table join that only needs the LIMIT fix.

**What the interviewer wants**: the realization that the LLM is an untrusted code generator — the guardrail layer is a contract, regex/AST validation is the mechanism, and BLOCK beats silent-rewrite for anything ambiguous.

In [0]:
# Solution Q4 — text-to-SQL guardrail validator + rewriter
import re

ALLOWED_TABLES = {
    "main.sales.gold_orders":    ["order_id","customer_id","amount","order_date","region"],
    "main.sales.gold_customers": ["customer_id","name","region","signup_date"],
}
FORBIDDEN_TOKENS = r"\b(INSERT|UPDATE|DELETE|MERGE|CREATE|DROP|ALTER|TRUNCATE|GRANT|REVOKE|COPY|EXPORT)\b"
MAX_LIMIT = 1000

def guard(sql):
    """Validate + rewrite LLM-generated SQL. BLOCK decisions mean: never execute, escalate to a human."""
    issues, fixed = [], sql.strip().rstrip(";")
    up = fixed.upper()
    if re.search(FORBIDDEN_TOKENS, up):
        issues.append(("BLOCK", f"write/DDL verb found: {re.search(FORBIDDEN_TOKENS, up).group(1)}"))
    tabs = [t.strip("`") for t in re.findall(r"(?:FROM|JOIN)\s+([a-zA-Z0-9_.`]+)", fixed, flags=re.I)]
    bad = [t for t in tabs if t not in ALLOWED_TABLES]
    if bad:
        issues.append(("BLOCK", f"table not in allowlist: {bad}"))
    if re.search(r"SELECT\s+\*", up):
        issues.append(("FIX", "SELECT * -> explicit column list (cost + PII governance)"))
        cols = ", ".join(ALLOWED_TABLES[tabs[0]][:4]) if tabs and tabs[0] in ALLOWED_TABLES else "/* columns */"
        fixed = re.sub(r"SELECT\s+\*", f"SELECT {cols}", fixed, count=1, flags=re.I)
    if not re.search(r"\bLIMIT\s+\d+", up):
        issues.append(("FIX", f"injected LIMIT {MAX_LIMIT}"))
        fixed = f"{fixed}\nLIMIT {MAX_LIMIT}"
    else:
        n = int(re.search(r"LIMIT\s+(\d+)", up).group(1))
        if n > MAX_LIMIT:
            issues.append(("FIX", f"LIMIT {n} clamped to {MAX_LIMIT}"))
            fixed = re.sub(r"LIMIT\s+\d+", f"LIMIT {MAX_LIMIT}", fixed, flags=re.I)
    execute = not any(kind == "BLOCK" for kind, _ in issues)
    return fixed, issues, execute

tests = [
    "SELECT * FROM main.sales.gold_orders WHERE region = 'EMEA'",
    "SELECT amount FROM main.sales.gold_orders LIMIT 50000",
    "SELECT order_id FROM main.ops.secret_payouts",
    "DELETE FROM main.sales.gold_orders",
    "SELECT order_id, amount FROM main.sales.gold_orders o JOIN main.sales.gold_customers c ON o.customer_id = c.customer_id",
]
for sql in tests:
    f, i, ex = guard(sql)
    verdict = "EXECUTE" if ex else "ABORT (escalate)"
    print(f"  {verdict:18s} issues={i}")
    print(f"  -> {f[:90]}")

f1, i1, e1 = guard(tests[0]); assert e1 and "order_id, customer_id" in f1 and len(i1) == 2
f2, i2, e2 = guard(tests[1]); assert e2 and "LIMIT 1000" in f2
f3, i3, e3 = guard(tests[2]); assert not e3 and any(k == "BLOCK" for k, _ in i3)
f4, i4, e4 = guard(tests[3]); assert not e4 and i4[0][1].startswith("write/DDL verb")
f5, i5, e5 = guard(tests[4]); assert e5 and len(i5) == 1 and "injected LIMIT" in i5[0][1]
print("Q4 OK: 5 adversarial cases -> block/fix/clamp/inject decisions all correct")
print()
print("Production notes: parse with sqlglot for a REAL AST (regex is a toy), run as a service principal")
print("with read-only grants, add cost caps via query tags, and log every rewrite for the audit trail.")

## Q5 — PySpark: Streaming Deduplication State Semantics — `dropDuplicates` vs `dropDuplicatesWithinWatermark` [HARD]

**Interviewer (Netflix):** We replay a Kafka topic that delivers at-least-once. The pipeline dedups on device id — but a device can legitimately emit events for hours. Engineering asks three questions and your job is to answer with **semantics, not vibes**:

1. `dropDuplicates(["key"])` — does the state store EVER forget a key (even with a watermark)?
2. What is the correct dedup key when the same entity emits many distinct events — and what does each choice drop or keep?
3. When is `dropDuplicatesWithinWatermark` the right tool, and what exactly does it guarantee (and NOT guarantee)?

**Setup (executed below)**: three files land one per micro-batch (`maxFilesPerTrigger=1`, `availableNow`):
- File A: `dev-1@10:00:10` (+ exact duplicate), `dev-2@10:02:00` (+ duplicate)
- File B: `dev-9@12:00:00` (a time-advancer that pushes the event-time watermark forward)
- File C: `dev-1@12:30:00` — **the same key, 2.5 hours later**

**Predict before running**: which variants emit `dev-1@12:30`? Then verify: the no-watermark and key-only-watermark variants SWALLOW it (key state never evicted), while `dropDuplicates(["key","event_time"])` emits it. Read the state-store metrics from `lastProgress` for the unbounded-state proof.

**What the interviewer wants**: you know dedup state is a state-store growth problem (memory, RocksDB, eviction), the exact eviction rules of each API, and that watermark ≠ dedup-state eviction for `dropDuplicates`.

In [0]:
# Solution Q5 — empirical streaming dedup semantics (re-runnable: unique paths per execution)
import uuid, time
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType

run = uuid.uuid4().hex[:8]            # fresh source + checkpoints each run -> deterministic results
src  = f"/Volumes/workspace/default/etl_test_volume/d22_q5_src_{run}"
base = f"/Volumes/workspace/default/etl_test_volume/d22_q5_ck_{run}"

fa = spark.createDataFrame([("dev-1","2026-11-04 10:00:10",1.0),("dev-1","2026-11-04 10:00:10",1.0),("dev-2","2026-11-04 10:02:00",3.0)],"key string, event_time string, val double").withColumn("event_time", F.to_timestamp("event_time"))
fb = spark.createDataFrame([("dev-9","2026-11-04 12:00:00",7.0)],"key string, event_time string, val double").withColumn("event_time", F.to_timestamp("event_time"))
fc = spark.createDataFrame([("dev-1","2026-11-04 12:30:00",9.0)],"key string, event_time string, val double").withColumn("event_time", F.to_timestamp("event_time"))
for d in [fa, fb, fc]:                      # flat files, one per micro-batch (files in subdirs are NOT seen)
    d.coalesce(1).write.mode("append").parquet(src)
    time.sleep(2)

schema = StructType([StructField("key", StringType()), StructField("event_time", TimestampType()), StructField("val", DoubleType())])
reader = (spark.readStream.format("parquet").schema(schema).option("path", src).option("maxFilesPerTrigger", 1))

variants = {
  "dd_nowm":    (reader.load(),                             ["key"]),                  # no watermark
  "dd_wm_key":  (reader.load().withWatermark("event_time","10 minutes"), ["key"]),        # watermark, key-only
  "dd_wm_kt":   (reader.load().withWatermark("event_time","10 minutes"), ["key","event_time"]),  # key+event_time
  "dd_wmw":     (reader.load().withWatermark("event_time","10 minutes"), "WITHIN"),        # dropDuplicatesWithinWatermark
}
progress = {}
for name, (src_df, keyspec) in variants.items():
    dd = src_df.dropDuplicatesWithinWatermark(["key"]) if keyspec == "WITHIN" else src_df.dropDuplicates(keyspec)
    q = (dd.writeStream.format("memory").queryName(f"{name}_{run}").outputMode("append")
           .option("checkpointLocation", f"{base}_{name}")
           .trigger(availableNow=True).start())
    q.awaitTermination()
    progress[name] = q.lastProgress

outs = {}
for name in variants:
    rows = sorted((r["key"], str(r["event_time"]), float(r["val"])) for r in spark.sql(f"SELECT * FROM {name}_{run}").collect())
    outs[name] = rows
    print(f"{name:10s} -> {rows}")

assert ("dev-1","2026-11-04 12:30:00",9.0) not in outs["dd_nowm"]    # never forgets the key: legit event swallowed
assert ("dev-1","2026-11-04 12:30:00",9.0) not in outs["dd_wm_key"]  # watermark does NOT evict dropDuplicates state
assert ("dev-1","2026-11-04 12:30:00",9.0) in outs["dd_wm_kt"]      # key+event_time treats it as a distinct event
print("Q5 OK: dedup-key choice, not the watermark, decides whether later same-key events survive")
print()
so = progress["dd_wm_key"]["stateOperators"][0]
print(f"state store at end of dd_wm_key: rows held = {so['numRowsTotal']}, rows evicted = {so['numRowsRemoved']}" )
print("-> dev-1's state is STILL there even though the watermark passed it by ~1h50m: dropDuplicates state is UNBOUNDED;")
print("   the watermark only drops too-late INPUT rows for it. At 10M active keys that is a RocksDB memory problem.")
print()
print("Semantics summary (empirical + docs):")
print("  dropDuplicates([key])       : key state kept FOREVER (watermark or not) -> unbounded state; later same-key events dropped.")
print("  dropDuplicates([key,ts])    : each distinct event is its own state row -> later same-key events EMITTED (still unbounded!).")
print("  dropDuplicatesWithinWatermark: REQUIRED watermark; dedups only while earliest/latest event distance < watermark delay;")
print("     duplicates outside the window are NOT guaranteed to be dropped, but state stays bounded. Use when the key can recur")
print("     forever (IoT devices) and re-delivery storms are short (e.g. 10-minute replay windows).")

## Q6 — PySpark: Build an IVF Vector Index from Scratch — ANN Search and the Probe/Recall Trade-off [FAANG]

**Interviewer (Databricks Vector Search):** You will NOT always have a managed vector index. Prove you understand what one does by building **IVF (inverted file index)** yourself on a Spark corpus:

1. **Corpus**: 600 normalized 3-D embeddings from 4 overlapping topics (real vector search normalizes and clusters — 3-D keeps it debuggable).
2. **Build**: k-means (k=4) → centroids are the IVF *cells*; publish `cell_id` per vector as the index.
3. **Query**: top-3 by cosine. Exact = brute force over all 600. IVF = rank cells by distance to the query, probe the top `n_cells` only (candidate pruning), score within candidates.
4. **Show the trade-off**: recall@3 at probe=1 vs probe=2 on 9 queries. With overlapping clusters, probe=1 must lose recall; probe=2 recovers it. Show the SQL-join pruning pattern (`JOIN cells ON ... WHERE cell IN (...)`).
5. Explain what production adds: HNSW graphs on top of IVF, quantization (PQ/SQ) to shrink vectors, and managed sync/refresh.

**What the interviewer wants**: ANN is an engineering trade-off (compute vs recall), not magic; candidate pruning = a join on cluster id; and honest recall measurement against brute force.

In [0]:
# Solution Q6 — IVF index build + probed ANN search vs brute force
import numpy as np
from pyspark.sql import functions as F, types as T

rng = np.random.default_rng(42)
centers = np.array([[0.9,0.1,0.1],[0.1,0.9,0.1],[0.1,0.1,0.9],[0.5,0.5,0.5]])
vecs = []
for c in centers:                              # overlapping clusters: sigma 0.3 spills across cell boundaries
    for _ in range(150):
        v = c + rng.normal(0, 0.3, 3)
        vecs.append(v / np.linalg.norm(v))
vecs = np.array(vecs)

corpus = spark.createDataFrame([(i, [float(x) for x in vecs[i]]) for i in range(len(vecs))],"id long, features array<double>")
corpus.createOrReplaceTempView("corpus")

# 1) build: k-means centroids = IVF cells
k, iters = 4, 20
cent = vecs[rng.choice(len(vecs), k, replace=False)]
for _ in range(iters):
    assign = np.argmin(((vecs[:, None, :] - cent[None, :, :])**2).sum(-1), axis=1)
    cent = np.array([vecs[assign == j].mean(axis=0) if (assign == j).any() else cent[j] for j in range(k)])
cell_of = np.argmin(((vecs[:, None, :] - cent[None, :, :])**2).sum(-1), axis=1)
print(f"IVF built: {len(vecs)} vectors, k={k} cells, {iters} k-means iterations")

# 2) publish the index: cell id per vector (the 'inverted file')
cell_map = spark.createDataFrame([(int(i), int(cell_of[i])) for i in range(len(vecs))], "id long, cell int")
cell_map.createOrReplaceTempView("cells")

def cosine(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
cos = F.udf(cosine, T.DoubleType())

def brute_force_topk(qv, n=3):
    sims = sorted(((int(i), cosine(qv, vecs[i])) for i in range(len(vecs))), key=lambda x: -x[1])
    return set(i for i, s in sims[:n])

def ivf_topk(qv, probes=1, n=3):
    dists = [((qv - cent[j])**2).sum() for j in range(k)]
    cells = sorted(range(k), key=lambda j: dists[j])[:probes]      # rank cells by distance to query
    cand = [i for i in range(len(vecs)) if cell_of[i] in cells]     # candidate pruning
    sims = sorted(((int(i), cosine(qv, vecs[i])) for i in cand), key=lambda x: -x[1])
    return set(i for i, s in sims[:n])

# 3) SQL pruning pattern: JOIN cells ON id WHERE cell IN (probe set), similarity via UDF
probe_cells = sorted(range(k), key=lambda j: ((vecs[0] - cent[j])**2).sum())[:2]
sql_top = (spark.sql("SELECT c.id, c.features FROM corpus c JOIN cells ce ON c.id = ce.id WHERE ce.cell IN (%s)"
                     % ",".join(map(str, probe_cells)))
           .withColumn("sim", cos("features", F.array(*[F.lit(float(x)) for x in vecs[0].tolist()])))
           .orderBy(F.desc("sim")).limit(3))
print("SQL-probed top-3:", sorted(r["id"] for r in sql_top.collect()), "| exact:", sorted(brute_force_topk(vecs[0])))

# 4) the trade-off, quantified
qids = [0, 150, 300, 450, 42, 100, 250, 400, 7]
r1 = np.mean([len(ivf_topk(vecs[i], 1) & brute_force_topk(vecs[i])) / 3 for i in qids])
r2 = np.mean([len(ivf_topk(vecs[i], 2) & brute_force_topk(vecs[i])) / 3 for i in qids])
print(f"recall@3: probe=1 -> {r1:.2f}, probe=2 -> {r2:.2f} (probing 2 of 4 cells = 2x compute for +{(r2-r1)*100:.0f}pp recall)")
assert r1 < 1.0 and r2 > r1, (r1, r2)
print("Q6 OK: candidate pruning trades compute for recall; production adds HNSW + PQ quantization on top of IVF")

## Q7 — Spark Optimization & Deep Spark: Shuffle Mechanics + Push-Based Shuffle [HARD]

**Interviewer (Netflix, Spark OSS contributor vibe):** Everyone tunes `spark.sql.shuffle.partitions`. Almost nobody can say what a shuffle **is**. Walk the full path of one wide transformation on 10 TB:

1. **Write path**: what does each map task emit, where does it land (local disk? what files?), and what does `spark.shuffle.spill` change? Sort-based writer: what is the shuffle file layout (one file per reducer? per map task with index?)
2. **Read path**: how does a reduce task find its blocks (driver metadata? direct fetch from map executors?), and what actually happens on a **fetch failure** — why does Spark re-run ONE map stage, and when does it give up (`spark.shuffle.io.maxRetries`, `spark.stage.maxConsecutiveAttempts`)?
3. **Push-based shuffle (Databricks 'Magnet')**: what changes when the map side eagerly merges blocks on remote shuffle services, and why does it cut tail latency + file count? When is it a loss?
4. **Tuning ladder**: shuffle partition sizing (2-3× core count, bytes/partition 100-200 MB), skew (AQE skew join vs salting), spills (fewer/larger partitions, `spark.shuffle.file.buffer`), and the Zero-shuffle goal (pipeline as much as possible).

Days 13/15 covered minimizing shuffle DATA and spill triage — this question is the **mechanism underneath**.

In [0]:
# Solution Q7 — shuffle mechanics talk track (configs verified against Spark defaults)
TALK_TRACK = [
  ("1. WRITE PATH (map task)",
   "Each map task evaluates its pipeline, then for each downstream partition it serializes + hashes key -> partition_id.\n"
   "   Sort-based shuffle (default): each map task writes ONE data file + ONE .index per map task, sorted by partition id\n"
   "   so a reducer can binary-search its offset. Rows exceeding spark.shuffle.sort.bypassMergeThreshold (~200 for\n"
   "   no-aggregation shuffles) force the full sort path. No reducer ever gets its own file - the index is the address.\n"
   "   If the sorter's memory (spark.shuffle.spill.* / ExecutionMemoryPool) fills, records spill: sorted runs on disk\n"
   "   that get merge-sorted later. Spill is CORRECTNESS-preserving, only slow."),
  ("2. READ PATH (reduce task)",
   "MapStage completion is registered with the driver (MapOutputTracker). Each reduce task asks the driver for block\n"
   "   locations/offsets and FETCHES directly from the map executors over Netty (spark.shuffle.io.* retries/backoff).\n"
   "   FETCH FAILURE: an executor died or its shuffle files vanished. Spark assumes the STAGE's outputs are suspect,\n"
   "   re-schedules the lost map stage (not the whole job) up to spark.stage.maxConsecutiveAttempts (default 4).\n"
   "   Abandoned shuffle files from killed executors are why long chains of stage retries smell like disk pressure."),
  ("3. PUSH-BASED SHUFFLE (Databricks Magnet)",
   "Classic shuffle: every reducer fetches from EVERY map task => with 10k maps x 10k reduces you get 100M small\n"
   "   RPC fetches, and stragglers amplify. Magnet: map tasks PUSH blocks to remote shuffle services which EAGERLY\n"
   "   MERGE them into per-reducer files while maps run. Reducers then fetch from a handful of merged files:\n"
   "   - far fewer open connections, sequential reads, less tail latency\n"
   "   - works because cloud storage/remote services are highly available and writes are idempotent-ish (offsets tracked)\n"
   "   Loss cases: tiny jobs (push overhead > merge benefit), tiny clusters with no remote services, and very small\n"
   "   partitions where the merge step is pure overhead. On Databricks it is on by default for eligible jobs."),
  ("4. TUNING LADDER (in order)",
   "(a) Eliminate: broadcast small sides, filter/partial-aggregate BEFORE the exchange (Days 13/15).\n"
   "   (b) Size partitions: aim 100-200 MB/partition, 2-3x total cores; AQE coalescing handles post-shrink sizing.\n"
   "   (c) Skew: spark.sql.adaptive.skewJoin.enabled (split + replicate hot partitions) BEFORE manual salting.\n"
   "   (d) Spills: larger spark.shuffle.file.buffer (1m), spark.shuffle.spill.compress on (default), then bigger partitions.\n"
   "   (e) Watch Spark UI: shuffle read/write per stage, remote vs local reads, fetch-failure counters - the shuffle\n"
   "       story of a job is readable from the stage graph alone."),
  ("5. INTERVIEW SOUND-BITE",
   "'A shuffle is a distributed external sort: maps pre-partition by hash, reducers fetch by offset. Everything\n"
   "   expensive - spills, stragglers, fetch failures - is a consequence of that file exchange. Tune the file\n"
   "   exchange, not the number."),
]
for topic, answer in TALK_TRACK:
    print(f"{topic}\n{'-'*len(topic)}")
    print(answer); print()
print("Q7 OK: talk track complete - write path, read path, Magnet, tuning ladder")

## Q8 — Spark Optimization: Lakehouse Federation — What Actually Pushes Down? [HARD]

**Interviewer (Databricks, cross-cloud customer):** We federate a 4 TB Snowflake/Postgres table via a UC foreign catalog. Analysts run BI on it and blame Spark when it is slow. You own the answer:

1. **Pushdown surface**: which operators does Spark push to the foreign source — filters, projections (column pruning), LIMIT, aggregations, joins, window functions? What almost never pushes (Spark-native UDFs, complex casts, some expressions)? How do you CHECK (not guess) — which part of `EXPLAIN FORMATTED` / `df.explain()` shows the `Federated`/`PushDownFilters` operators?
2. **The table-scan cliff**: when does Spark fall back to `DirectScan` (pulling whole tables)? Why is `SELECT *` on a federated table catastrophic, and what does the JDBC batch size do?
3. **Design patterns**: copy-in (materialize hot tables on a schedule vs query-through), connection pooling, foreign catalog caching of metadata, partitioning/predicate design so the source can use ITS indexes (avoid functions wrapping join/filter columns).
4. **Cost model**: when does query-through beat copy-in (freshness, infrequent access), and when must you materialize?

**What the interviewer wants**: federation is an optimization CONTRACT — you verify what crossed the wire, you design predicates so the foreign engine does the work, and you know when to stop federating.

In [0]:
# Solution Q8 — federation pushdown talk track + verification patterns
TALK_TRACK = [
  ("1. WHAT PUSHES DOWN",
   "Spark Connect's federation layer builds a 'pushdown query' for each scan. Pushed reliably:\n"
   "   - Projection / column pruning (SELECT a, b -> only those columns cross the wire)\n"
   "   - Simple filters on native types =, <, >, IN, LIKE, simple casts, IS NULL\n"
   "   - Aggregations and even GROUP BY in recent runtimes (pushable aggregate rules)\n"
   "   - LIMIT (pushed as TOP/limit in the source dialect)\n"
   "   Rarely/never pushed: Python/Scala UDFs (black box to the source), non-deterministic expressions,\n"
   "   complex date/string functions, window functions, anything the dialect translator cannot render.\n"
   "   Multi-source joins NEVER push one source into another; a join of two foreign tables of the SAME\n"
   "   catalog/connection can push; otherwise Spark joins locally after two scans."),
  ("2. VERIFY, DON'T GUESS",
   "df = spark.table('snowcat.analytics.orders').filter(col('order_date') > '2026-10-01').select('order_id','amount')\n"
   "   df.explain(True)  -> look for the scan node: a Federated scan shows PushedFilters/PushedProjection in the\n"
   "   physical plan. If you see a bare DirectScan with no filters -> the WHOLE table (or whole partition scan) is\n"
   "   being pulled and Spark filters row-by-row. Query the source's own query history (Snowflake QUERY_HISTORY,\n"
   "   pg_stat_statements) and CONFIRM the SQL the source actually received - the ultimate ground truth."),
  ("3. THE CLIFFS",
   "- SELECT * : projects nothing -> full row payload over JDBC/HTTP; columnar sources degrade to row fetch.\n"
   "   - Functions around filter columns: WHERE upper(email) = ... blocks predicate pushdown on the SOURCE index.\n"
   "   Push raw literals, transform later.\n"
   "   - Tiny fetch batches (spark.sql.federation.fetchsize / JDBC fetchsize default ~100-1000) -> many round trips;\n"
   "   raise for wide scans.\n"
   "   - Cross-join-ish plans: if Spark cannot push the join, BOTH sides scan fully and join locally - usually the\n"
   "   moment you materialize instead."),
  ("4. COPY-IN vs QUERY-THROUGH",
   "Query-through wins: freshness-critical (minutes), low QPS, selective filters, source licensed for the load.\n"
   "   Copy-in wins: BI dashboards (repeated full scans), ML training (needs Spark-native data), anything with\n"
   "   UDF-heavy transforms, or source egress/compute costs > storage costs.\n"
   "   Pattern: schedule a materialize step (CREATE TABLE ... AS SELECT from foreign table with a pushed predicate\n"
   "   + incremental watermark) behind the same contract, and the BI layer never knows. Lakeflow Connect/managed\n"
   "   ingestion automates exactly this loop."),
  ("5. SOUND-BITE",
   "'Federation pushdown is a contract with the foreign engine: verify the SQL it received, keep predicates\n"
   "   literal so its indexes work, and the moment the wire cost beats the storage cost - materialize.'"),
]
for topic, answer in TALK_TRACK:
    print(f"{topic}\n{'-'*len(topic)}")
    print(answer); print()
print("Q8 OK: pushdown surface, verification, cliffs, copy-in decision rule")

## Q9 — ETL: GenAI Document Ingestion at Scale — Idempotent Bronze + DLQ Retry Budget + Circuit Breaker [FAANG]

**Interviewer (Adobe, AI platform team):** We ingest 50k PDFs/docx daily into a RAG corpus. Re-runs must NOT re-parse/re-embed unchanged files (embeddings are the biggest line item), a corrupt file must not poison the batch, and a stuck source must not burn money overnight.

**Requirements**
1. **Idempotent bronze**: key the MERGE on `(path, content_sha)` — unchanged content → skip (zero re-embeds); changed content → `content_version+1`; new → insert. Prove a full re-delivery of the same batch re-embeds 0 files.
2. **Poison isolation**: unparseable files go to a DLQ table with an `attempts` counter — the batch still succeeds.
3. **Circuit breaker**: after 3 consecutive failures on a source, mark the circuit OPEN and skip (log, don't process) until a reset. Re-run the failing batch 4 times and show the breaker tripping.

**What the interviewer wants**: content-hash idempotency (not path idempotency!), retry budgets instead of infinite retries, and the admission that the circuit breaker (Day 16) here guards COST, not correctness.

In [0]:
# Solution Q9 — unstructured ingestion: idempotent MERGE + DLQ + circuit breaker (run-unique paths)
import hashlib, uuid
from pyspark.sql import functions as F

run = uuid.uuid4().hex[:8]
base = f"/Volumes/workspace/default/etl_test_volume/d22_q9_{run}"

docs = [
    {"path": "s3://docs/contracts/v1.pdf", "text": "Master agreement terms...", "parse_ok": True},
    {"path": "s3://docs/invoices/i7.pdf",  "text": "Invoice #7 total 1200...", "parse_ok": True},
    {"path": "s3://docs/reports/bad.docx", "text": "corrupt-binary",           "parse_ok": False},
    {"path": "s3://docs/reports/r2.txt",  "text": "Q3 revenue summary...",  "parse_ok": True},
]
def sha(t): return hashlib.sha256(t.encode()).hexdigest()[:16]

spark.createDataFrame([], "path string, content_sha string, content_version int, ingest_run int") \
     .write.mode("error").format("delta").save(f"{base}/bronze")
spark.createDataFrame([("s3://docs/reports/bad.docx", sha("corrupt-binary"), 0)],
                      "path string, content_sha string, attempts int") \
     .write.mode("error").format("delta").save(f"{base}/dlq")

def load_ingest(new_batch):
    """MERGE on path; changed sha -> bump version. Returns (reembedded, skipped) vs PRE-MERGE snapshot."""
    before = {(r["path"], r["content_sha"]) for r in spark.read.format("delta").load(f"{base}/bronze").collect()}
    b = spark.createDataFrame([(d["path"], sha(d["text"])) for d in new_batch if d["parse_ok"]],
                              "path string, new_sha string")
    b.createOrReplaceTempView("incoming")
    spark.sql(f"""
      MERGE INTO delta.`{base}/bronze` t USING incoming s ON t.path = s.path
      WHEN MATCHED AND t.content_sha <> s.new_sha THEN
        UPDATE SET content_sha = s.new_sha, content_version = t.content_version + 1
      WHEN NOT MATCHED THEN
        INSERT (path, content_sha, content_version, ingest_run) VALUES (s.path, s.new_sha, 1, 1)
    """)
    reembedded = sum(1 for d in new_batch if d["parse_ok"] and (d["path"], sha(d["text"])) not in before)
    skipped    = sum(1 for d in new_batch if d["parse_ok"] and (d["path"], sha(d["text"])) in before)
    return reembedded, skipped

def process_dlq(new_batch, circuit_open):
    out = []
    for d in new_batch:
        if not d["parse_ok"]:
            if circuit_open:
                out.append((d["path"], "SKIPPED - circuit OPEN (3 consecutive failures)"))
                continue
            spark.sql(f"""
              MERGE INTO delta.`{base}/dlq` t
              USING (SELECT '{d['path']}' AS path) s ON t.path = s.path
              WHEN MATCHED THEN UPDATE SET attempts = t.attempts + 1
            """)
            atts = [r["attempts"] for r in spark.read.format("delta").load(f"{base}/dlq").collect() if r["path"] == d["path"]][0]
            out.append((d["path"], f"attempt {atts}" + (" - TRIP! next failure opens circuit" if atts >= 2 else "")))
    return out

circuit_open = False
for run_no in range(1, 5):
    r = load_ingest(docs)
    d = process_dlq(docs, circuit_open)
    atts = [r2["attempts"] for r2 in spark.read.format("delta").load(f"{base}/dlq").collect()][0]
    circuit_open = atts >= 3
    print(f"run {run_no}: re-embedded={r[0]} skipped={r[1]} | DLQ: {d} | circuit_open={circuit_open}")

rows = {(r["path"], r["content_sha"], r["content_version"]) for r in spark.read.format("delta").load(f"{base}/bronze").collect()}
assert len(rows) == 3
atts = [r["attempts"] for r in spark.read.format("delta").load(f"{base}/dlq").collect()][0]
assert atts == 3
print("Q9 OK: first delivery embeds 3, re-deliveries re-embed 0 (content-hash idempotency),")
print("     DLQ attempts accumulate 1->2->3, then the breaker opens and skips further work (cost guard).")

## Q10 — ETL: Cross-Cloud Disaster Recovery for a Delta Lakehouse [HARD]

**Interviewer (Uber, platform team):** Regulation forces half our workloads to run in a second cloud within 30 days of a regional failure. We have 800 Delta tables in Unity Catalog, streaming pipelines on the primary, and a 15-minute RPO for tier-1 tables. Design the DR story. (Days 18/19/20 covered Snowflake replication, DEEP CLONE backups, and chaos game-days — this question is the **multi-cloud architecture** layer on top.)

**Requirements**
1. **Replication topology options**: cloud-native storage replication (S3 CRC / ADLS RA-GRS / GCS dual-region), Delta DEEP CLONE on schedule, and Delta Sharing read-availability — compare RPO/RTO, cost, and consistency for each tier.
2. **The metadata problem**: replicated data is useless if the CATALOG, table IDs, permissions, and credentials do not come with it. What must you replicate and RE-MAP (external locations, storage credentials, service principals, UC grants)? Why is `delta.log` consistency the subtle part (no atomic cross-cloud rename)?
3. **Pipelines in DR**: how do streaming checkpoints behave on failover (offsets point at primary source shards — Kafka MirrorMaker / offset translation)? When do you switch to the replica table ids?
4. **Runbook + verification**: quarterly failover game-day, read-only warm standby for tier-1 (PROOF of RTO), tier-2 = rebuild-from-source, tier-3 = archive only. End with the RPO/RTO table you would present to the CTO.

**What the interviewer wants**: DR is a METADATA problem more than a data problem; honest tiering (you cannot give everything 15-minute RPO); and verification that a failover actually works — not a document that says it does.

In [0]:
# Solution Q10 — cross-cloud DR talk track + decision matrix
DR_MATRIX = [
  ("Tier-1 gold (payments, fraud)", "15 min", "2 h",
   "Continuous storage replication + warm standby UC catalog; pipelines can resume from mirrored Kafka"),
  ("Tier-2 silver/analytics", "24 h", "24 h",
   "Nightly DEEP CLONE of tables to the DR cloud storage account + catalog-as-code rebuild"),
  ("Tier-3 bronze/archive", "best effort", "days",
   "Replay from source systems / archive only; no compute standing in DR"),
  ("Streaming checkpoints", "n/a", "n/a",
   "Offsets reference PRIMARY Kafka shards: run MirrorMaker/replicated topics in DR and REWRITE checkpoint"
   " metadata at failover, or (simpler) restart pipelines at a known timestamp with idempotent MERGE sinks"),
  ("Catalog + grants", "n/a", "n/a",
   "UC metastore is per-region: replicate as CODE (Terraform/DABs) - schemas, grants, external locations,"
   " storage credentials with DR-cloud principals. Data without grants is unreadable at 3 AM."),
  ("delta.log consistency", "n/a", "n/a",
   "THE subtle part: Delta commits are a single atomic rename on the primary. Async replication delivers _log"
   " files out of order; a replica may see a newer commit file before an older one - treat the replica as"
   " possibly-N-seconds-behind and NEVER write to it before failover (split-brain guard: flip a DNS/flag first)."),
]
print(f"{'Component':34s} {'RPO':10s} {'RTO':6s} Strategy")
print("-"*110)
for name, rpo, rto, strat in DR_MATRIX:
    print(f"{name:34s} {rpo:10s} {rto:6s} {strat}")
print()
print("Runbook skeleton (the part interviewers actually grade):")
print("  1. DECLARE: incident commander flips the failover flag (write-access is cut to primary immediately).")
print("  2. CATALOG: apply Terraform/DAB for schemas+grants on the DR metastore (10 min, automated).")
print("  3. VALIDATE: read-only smoke queries on tier-1 replica tables (row counts vs last replicated version).")
print("  4. PROMOTE: flip external-location DNS/credential to the replicated storage; re-point pipelines with")
print("     idempotent sinks; restart streams from mirrored topics at a known offset.")
print("  5. VERIFY tier-1 RTO publicly (a stopwatch, not a slide). Quarterly game-day makes step 5 boring.")
print()
print("Q10 OK: tiered RPO/RTO matrix, metadata-as-code, checkpoint/offset story, split-brain guard, runbook")

## Q11 — ETL: Streaming Tables vs Materialized Views in the Lakehouse [HARD]

**Interviewer (Databricks, Lakeflow):** Since Spark Declarative Pipelines (SDP, formerly DLT), the lakehouse has THREE declarative table types and a lot of confusion. Explain them like you are writing the internal design doc:

1. **Semantics**: Streaming Table (ST) vs Materialized View (MV) vs plain view — append-only event-time source vs fully-refreshable query result. What may an ST read (Auto Loader/Kafka/CDC sources, other STs), what may an MV read (any table, including non-SDP)?
2. **Refresh modes**: incremental processing vs complete refresh; `CREATE OR REFRESH STREAMING TABLE`; backfills and `REFRESH` with `full_refresh=True` — when each fires and what the pipeline graph guarantees (topological order, once-per-refresh).
3. **The late/backfill story**: what happens when historical files arrive late to an ST; when an MV is the WRONG tool (O(n) full recompute per refresh); row-level changes from CDC with `apply_changes()` into an ST.
4. **Kappa vs Lambda**: how the streaming-table pattern implements Kappa (one code path, replayable source) and when you still end up with a Lambda (batch corrections + speed layer).
5. Decide: clickstream events into a sessionized gold table (dedup + watermark) — ST, MV, or both, and why?

**What the interviewer wants**: refresh semantics you can reason about like a query planner (what is incremental, what is re-computed, what is replayed), and the judgment to pick table types per access pattern.

In [0]:
# Solution Q11 — streaming tables vs materialized views talk track
TALK_TRACK = [
  ("1. SEMANTICS",
   "STREAMING TABLE: defined ONCE over an append-only source (Auto Loader files, Kafka, CDF, or another ST) -"
   "   the pipeline processes each batch of new data exactly once and APPENDS/merges into the table. It carries"
   "   event-time semantics (watermarks, windows) because the source is a stream.\n"
   "   MATERIALIZED VIEW: a query result kept fresh - the engine decides what to recompute, and MAY do a full"
   "   refresh when it cannot incrementally maintain the query (non-incrementable aggregates, complex joins)."
   "   Reads ANY tables (SDP or not) but has no stream semantics.\n"
   "   Plain VIEW: a saved query, zero storage, recomputed at every read."),
  ("2. REFRESH MECHANICS",
   "A pipeline update walks the graph in topological order: sources -> STs -> downstream STs/MVs, each once per"
   "   update. CREATE OR REFRESH STREAMING TABLE t = full refresh of THAT table only (re-ingests everything the"
   "   source still exposes); a scoped REFRESH can force backfills. MVs refresh with the pipeline; a full refresh"
   "   of an MV is O(table size) - fine for thousands of rows, terrifying for billions."),
  ("3. LATE DATA + CDC",
   "STs: late files arrive in the NEXT update (that is fine - files are the source of truth); late EVENT TIMES"
   "   inside records are a watermark problem (Day 16). dropDuplicates/state ops run per-update.\n"
   "   CDC into an ST via apply_changes()/AUTO CDC: the pipeline applies inserts/updates/deletes against the"
   "   target with sequence columns - your ST behaves like a synchronized replica, not an append log.\n"
   "   MV trap: a source table with row-level updates forces MV recompute proportional to the change if the"
   "   aggregate cannot be incrementally maintained - check the incrementalization insights before shipping."),
  ("4. KAPPA VS LAMBDA",
   "Kappa = ONE (replayable) code path: raw events -> ST bronze -> ST gold; corrections handled by reprocessing"
   "   from the source with new code (CREATE OR REFRESH). The streaming-table pattern IS kappa when the source"
   "   can be replayed (Auto Loader directories, Kafka retention).\n"
   "   You drift back to Lambda when: batch-only sources feed the gold layer, or corrections come from a"
   "   separate backfill pipeline writing the same tables - name that drift explicitly."),
  ("5. THE CLICKSTREAM DECISION",
   "Bronze raw clicks: ST (append-only source, idempotent file ingestion).\n"
   "   Gold sessionized/deduped table: ST with withWatermark + stateful dedup (dropDuplicatesWithinWatermark -"
   "   Day 22 Q5!) - sessions are event-time objects, MV cannot express them.\n"
   "   Daily active-user rollups for BI: MV over the gold ST - refreshes incrementally per update, and BI gets"
   "   a table, not a stream. Both types in one pipeline is the NORMAL answer, not a compromise."),
]
for topic, answer in TALK_TRACK:
    print(f"{topic}\n{'-'*len(topic)}")
    print(answer); print()
print("Q11 OK: ST vs MV vs view semantics, refresh/backfill, CDC, kappa/lambda, clickstream decision")

## Q12 — Data Modeling: The RAG Corpus as a Governed Data Product [FAANG]

**Interviewer (Microsoft, Copilot data team):** Our RAG corpus is 40M chunks from 200k documents across 6 teams. Retrieval quality regressions are untraceable, embeddings get re-generated silently, and nobody can answer "what would change if we re-embedded with model v2?". Model the corpus as a **governed data product**.

**Requirements — design the tables + rules:**
1. **Chunk provenance**: `bronze_documents` (path, content_sha, source, ingest_run) → `silver_chunks` (chunk_id, doc_sha, ordinal, text, chunking_config) → `gold_embeddings` (chunk_id, embedding_model_version, embedding, vector_index_sync_state). Why must `chunk_id` be deterministic (hash of doc_sha + ordinal + chunking_config)? Why does `content_sha` on the DOC level (not chunk level) drive idempotency (Day 22 Q9)?
2. **Embedding versioning**: pin `embedding_model_version` on every row; never overwrite embeddings in place. A model upgrade = new rows + a new index build + a shadow evaluation — never a silent UPDATE.
3. **Lineage + reproducibility**: given an answer that cited chunk X, you must be able to reconstruct exactly which document version, chunker config, and embedding model produced it. What columns make that query possible? (Day 15 built the idempotent MERGE; this question is the SCHEMA that makes it governable.)
4. **Evaluation tables**: a `retrieval_evals` table (query, retrieved_chunk, rank, relevance, eval_config) — why is eval data a first-class product table, and how does it gate index promotion (compare Day 21's RAG metrics: P@k, MRR, nDCG)?
5. **Contract**: what goes in the data contract (chunker config immutability, embedding model deprecation policy, freshness SLA, PII flags per source)?

**What the interviewer wants**: embeddings are DATA with lineage, not artifacts; version-pin everything; and eval data gates promotion the same way tests gate code.

In [0]:
# Solution Q12 — RAG corpus data product: DDL sketch + governance rules (design question)
DDL = [
  ("bronze_documents", [
    "path STRING, content_sha STRING, source_system STRING, ingest_run BIGINT, ingested_at TIMESTAMP,",
    "pii_flag BOOLEAN, PRIMARY KEY(path)",
    "-- content_sha (doc-level) drives idempotency: same sha -> skip re-parse (Q9); new sha -> new version"],),
  ("silver_chunks", [
    "chunk_id STRING PRIMARY KEY,  -- sha256(doc_sha || ordinal || chunking_config): DETERMINISTIC, so a",
    "  -- re-run rebuilds the same chunk ids -> MERGE is idempotent, lineage is stable across re-ingests",
    "doc_sha STRING, ordinal INT, text STRING, chunking_config STRING, created_at TIMESTAMP"],),
  ("gold_embeddings", [
    "chunk_id STRING, embedding_model_version STRING, embedding ARRAY<FLOAT>, index_sync_state STRING,",
    "computed_at TIMESTAMP, PRIMARY KEY (chunk_id, embedding_model_version)",
    "-- NEVER overwrite: model v2 = NEW rows; old rows stay queryable for reproducibility until deprecation"],),
  ("retrieval_evals", [
    "eval_run_id STRING, query STRING, retrieved_chunk_id STRING, rank INT, relevant BOOLEAN,",
    "eval_config STRING, embedding_model_version STRING, index_version STRING, judged_at TIMESTAMP",
    "-- P@k / MRR / nDCG (Day 21) computed per (eval_run, model_version, index_version) -> the promotion gate"],),
]
for name, cols in DDL:
    print(f"{name}:")
    for c in cols:
        print(f"  {c}")
    print()

RULES = [
  ("Deterministic chunk_id",
   "sha(doc_sha + ordinal + chunking_config) means: re-running the pipeline rebuilds IDENTICAL ids, so MERGE",
   "   is idempotent, lineage/eval history stays attached, and diffing corpora is a join, not a mystery."),
  ("Version-pinned embeddings",
   "Every row carries embedding_model_version. Upgrade = append v2 rows, build a SHADOW index, run the eval",
   "   harness, promote only if P@k/MRR/nDCG hold. Silent UPDATE would orphan every past citation + eval."),
  ("Reproducibility query",
   "answer -> cited chunk_id -> silver_chunks (doc_sha, chunking_config) -> bronze (path, content_sha, version)",
   "   -> 'exactly which bytes, chunker, and model produced this answer' in ONE join chain."),
  ("Contract essentials",
   "chunking_config immutability per doc_sha (changing the chunker = new doc_sha), model deprecation policy",
   "   (N-month overlap), freshness SLA per source_system, pii_flag propagation into the vector index (and",
   "   access-filtered retrieval - an index is a governed surface too, not just the table)."),
]
for topic, *lines in RULES:
    print(f"{topic}:")
    for ln in lines:
        print(f"  {ln}")
    print()
print("Q12 OK: four-layer corpus schema, deterministic ids, version pinning, eval-gated promotion, contract")

## Q13 — Data Modeling: Time-Series Metrics Platform Data Model [HARD]

**Interviewer (Netflix, Observability platform):** Every team wants dashboards of their service metrics. The naive design — one wide table per team, sampled every 10s — melts at 10M active time series. Design the data model for a metrics platform (the Prometheus/M3/Graphite problem, on a lakehouse):

1. **The narrow sample table**: why `(series_id, ts, value)` and NOT wide (`host, service, ... 200 metric columns`)? What is `series_id` — and what is the labels/tags dimension table that keeps cardinality explosion VISIBLE and governed?
2. **Cardinality budget**: `service x env x region x endpoint x status` — compute the series count formula, and design the guardrail (per-team series budget, label-value allowlists, cardinality monitoring on the labels table).
3. **Rollups/downsampling**: raw 10s samples → 1m → 10m → 1h rollup tables with (count, sum, min, max, last) — why sum+count (re-aggregatable averages) instead of pre-computed averages only? Which rollup does the 30-day dashboard read?
4. **Retention tiers**: raw 7 days → 1m/90d → 1h/2y → archive. How do you implement tiering (partition by day + scheduled moves / VACUUM interplay) and what does the query router do (pick the coarsest table that answers at the requested resolution)?
5. **Lakehouse angle**: why liquid clustering on `(series_id, ts)` beats ZORDER here; incremental rollup via streaming tables; and the late-arrival rule for samples.

**What the interviewer wants**: you treat cardinality as a first-class cost, rollups as lossless-where-possible, and the router as the piece that makes tiering invisible to users.

In [0]:
# Solution Q13 — metrics platform data model (DDL + cardinality math + router)
DDL = [
  ("labels (the series dimension)", [
    "series_id BIGINT PRIMARY KEY,   -- hash of the label SET: deterministic, one row per distinct series",
    "team STRING, service STRING, env STRING, region STRING, endpoint STRING, status STRING,",
    "metric_name STRING, created_at TIMESTAMP",
    "-- the cardinality of THIS table is the platform's real cost. It is visible, grantable, and alertable."],),
  ("samples_raw (10s)", [
    "series_id BIGINT, ts TIMESTAMP, value DOUBLE",
    "-- liquid clustered by (series_id, ts); partitioned by day; retention 7 days",
    "-- narrow beats wide: new metric = new label value, NOT a schema migration; 200-column-wide tables",
    "-- also destroy compression and make every dashboard query scan all columns"],),
  ("rollup_1m / rollup_10m / rollup_1h", [
    "series_id BIGINT, bucket_ts TIMESTAMP, cnt BIGINT, sum DOUBLE, mn DOUBLE, mx DOUBLE, last DOUBLE",
    "-- sum+cnt so averages RE-AGGREGATE losslessly across coarser buckets; mn/mx are only approximable",
    "-- downstream, and 'last' supports gauge semantics. A pre-averaged avg column CANNOT be re-averaged."],),
]
for name, cols in DDL:
    print(f"{name}:")
    for c in cols:
        print(f"  {c}")
    print()

# cardinality math: the guardrail interviewers want you to DO, not describe
teams = [
  ("payments", ["prod","staging"], ["us","eu","apac"], ["charge","refund","webhook"], ["2xx","4xx","5xx"]),
  ("search",    ["prod"],         ["us","eu"],     ["q","index","suggest"],    ["2xx","4xx","5xx"]),
]
total = 0
for team, envs, regions, endpoints, statuses in teams:
    n_series = len(envs) * len(regions) * len(endpoints) * len(statuses)
    total += n_series
    print(f"{team:10s}: {len(envs)} env x {len(regions)} region x {len(endpoints)} endpoint x {len(statuses)} status = {n_series} series")
print(f"total series: {total} (sample rows at 10s = {total * 6 * 24 * 365:,}/year from just {len(teams)} teams!)")
print("  guardrail: per-team series budget + label-value allowlist + a daily alert on labels-table growth")
print("  (cardinality explosion is a LABELS-table problem first, a samples problem second)")
print()

ROUTER = [
  ("last 6h, 10s resolution", "samples_raw", "ts > now()-6h", "exact"),
  ("last 30d dashboard",     "rollup_1m",  "bucket_ts > now()-30d", "1-min buckets re-averaged from sum/cnt"),
  ("last 2y capacity trend",  "rollup_1h",  "bucket_ts > now()-2y",  "hourly; 1m table already expired"),
]
print("query router (pick the COARSEST table that satisfies the requested resolution & range):")
for q, table, filt, how in ROUTER:
    print(f"  {q:26s} -> {table:12s} ({how})")
print()
print("retention tiers: raw 7d -> 1m/90d -> 1h/2y -> archive; implemented as daily partitions + scheduled")
print("moves/deletes (VACUUM after expiry, keep retention >= 7 days for time-travel on the rollups),")
print("late samples rule: samples older than the rollup bucket watermark are DROPPED on ingest (logged).")
print("Q13 OK: narrow samples + labels dimension, cardinality budget, lossless rollups, router, tiering")

## Q14 — Snowflake: Zero-Trust Security Stack — Network, Keys, Identity [HARD]

**Interviewer (Snowflake PS, bank customer):** Row access policies and masking (Day 16) protect data IN the account. The CISO asks about everything AROUND it: the perimeter. Walk the full zero-trust stack for a Snowflake deployment:

1. **Network isolation**: PrivateLink (AWS/Azure/GCP) — what traffic actually bypasses the public internet (Snowflake access AND external functions/stage egress)? Network policies (allowed/blocked IP lists, per-user), DNS resolution, and why a blocked network policy is the first thing to check on sudden 403s.
2. **Key management / encryption**: Tri-Secret Secure (EKMS + customer-managed key) — what the hierarchy is (root key rotates, tables derive file keys), what revoking YOUR key does (data unreadable, billing continues!), and the BYOK failure modes.
3. **Identity**: SCIM provisioning (Okta/Entra) — group sync vs direct grants, SSO (SAML/OAuth) + MFA, service accounts (key-pair auth, not passwords), and the de-provisioning SLA (disable on termination — SCIM gives you the lever).
4. **Least-privilege RBAC ladder**: functional roles (analyst_read_gold) vs ownership roles, future grants in schemas, and why `ACCOUNTADMIN` must be break-glass only.
5. Tie it together: the audit question — prove to an auditor that a specific user could not have exfiltrated a PII table (network policy + no share/export + masking + access history — Day 22 Q18 pattern).

**What the interviewer wants**: perimeter → keys → identity → grants → proof, in that order, and knowing each control's failure mode.

In [0]:
# Solution Q14 — Snowflake zero-trust security stack talk track
TALK_TRACK = [
  ("1. PRIVATELINK + NETWORK POLICIES",
   "PrivateLink gives a VPC-endpoint hostname: traffic to Snowflake stays on the provider backbone - no public",
   "   internet, no egress rules needed. CRITICAL detail: external stages/functions still egress separately -",
   "   block that with PrivateLink + dedicated egress or AWS PrivateLink for the stage too.\n"
   "   NETWORK POLICY: allowed_ip_list / blocked_ip_list per account, PER USER (service accounts can be pinned",
   "   to the CI subnet). They fire BEFORE authentication - sudden 403s for a whole office = check the policy",
   "   first (it is the outermost control, so it fails first and loudest)."),
  ("2. TRI-SECRET SECURE / EKMS",
   "Hierarchy: your KMS key + Snowflake's key -> account root key (rotated ~monthly) -> per-table file keys.",
   "   You hold one leg: revoke/destroy your KMS key and Snowflake CANNOT read the data (cryptographic kill",
   "   switch). Failure modes: KMS key disabled -> account effectively read-only/blocked while billing continues;",
   "   KMS region outage -> same; so key hygiene (alarms on KMS denies) is production-critical, not paperwork."),
  ("3. IDENTITY: SCIM + SSO + SERVICE ACCOUNTS",
   "SCIM (Okta/Entra) syncs users AND GROUPS - grant Snowflake roles to GROUPS, never to individuals, so",
   "   joiner/mover/leaver is handled in the IdP, not by a DBA. SSO via SAML/OIDC + MFA for humans. Service",
   "   accounts: key-pair auth (rotatable, no password in configs), a dedicated network policy (CI subnet),",
   "   and ONE owner + expiry review. De-provisioning SLA: disable at termination via SCIM - then validate that",
   "   disabled actually means 'cannot authenticate' (network policy as backstop)."),
  ("4. RBAC LADDER",
   "ACCOUNTADMIN: break-glass ONLY (vaulted, alerted). SECURITYADMIN: grants. SYSADMIN: objects. Then",
   "   FUNCTIONAL roles granted to IdP groups: e.g. analyst_read_gold = USAGE on schema + SELECT on future",
   "   tables in that schema (FUTURE GRANTS stop the 'new table, everyone re-grant' churn). Ownership lives in",
   "   deployment roles (Terraform/dbt), not in humans. The test: every human role answers 'what can you NOT",
   "   read' as crisply as what you can."),
  ("5. THE AUDIT ANSWER",
   "'Could user X have exfiltrated PII_TABLE on Nov 3?' -> (1) access history / LOGIN history: no query touched it",
   "   or every read was masked for their role; (2) network policy: their subnet could not reach a non-allowed",
   "   egress path; (3) no outbound share/pipe/export on the table; (4) the masking + row-access policy attached.",
   "   Four independent controls, each with evidence - that is what zero-trust means in an audit: not 'we trust", "   nothing', but 'we can PROVE each layer held'."),
]
for topic, *lines in TALK_TRACK:
    print(f"{topic}\n{'-'*len(topic)}")
    for ln in lines:
        print(ln)
    print()
print("Q14 OK: perimeter -> keys -> identity -> grants -> audit-proof, with failure modes")

## Q15 — Snowflake: Snowpark Container Services — GPU RAG Serving Inside the Account [HARD]

**Interviewer (Netflix/Snowflake partner architect):** We need to serve an open-source embedding + reranker model (GPU) for RAG, with ZERO data egress (regulation says embeddings of customer text must not leave the account). Snowpark Container Services (SCS) is on the table. Explain it and its alternatives:

1. **What SCS is**: OCI containers running INSIDE your Snowflake account (image repository in Snowflake, compute pools of GPU/CPU nodes, services/endpoints with REST access). Walk the pieces: `CREATE COMPUTE POOL`, `CREATE IMAGE REPOSITORY`, `CREATE SERVICE` — and what 'services vs jobs' means (long-running inference endpoint vs batch container).
2. **Why containers in the account matters**: the data-privacy argument (text never egresses; embeddings land directly in Snowflake tables), vs external API calls (egress, PII in prompts, latency, per-token cost).
3. **The RAG pattern end-to-end**: stage docs → SCS job chunks + embeds with a GPU model → embeddings table (Day 22 Q12 schema) → Cortex/vector or custom ANN in a service → query-time embedding + rerank in the same service. Where do you still use Cortex functions vs open weights (cost/control/fine-tune trade-off)?
4. **Operations**: billing (compute pool hours), scaling (min/max services), autoscaling limitations vs a true k8s cluster, image repo + CI/CD, and the failure modes (cold starts, pool starvation, GPU node availability).
5. **Decision table**: Cortex managed functions vs SCS open-weights vs external API vs data-egress + external GPU infra (Databricks model serving) — pick per scenario (regulation, fine-tuning, throughput, ops maturity).

**What the interviewer wants**: you know the 2024–2026 serving landscape on Snowflake, can argue privacy/cost/ops trade-offs, and you do NOT claim SCS is a general k8s replacement.

In [0]:
# Solution Q15 — Snowpark Container Services talk track
TALK_TRACK = [
  ("1. THE PIECES",
   "SCS runs OCI containers on Snowflake-managed compute INSIDE your account.\n"
   "   CREATE IMAGE REPOSITORY chatbot_db.schema.images;  -- docker push chatbot_registry Host/images/embedder:v3\n"
   "   CREATE COMPUTE POOL gpu_pool WITH (instance_family = 'GPU_NV_L');  -- Snowflake bills pool node-hours\n"
   "   CREATE SERVICE embed_service IN COMPUTE POOL gpu_pool FROM SPECIFICATION $$ ... $$  -- k8s-style spec\n"
   "   SERVICES = long-running HTTP endpoints (autoscaled, min/max instances); JOBS = run-to-completion\n"
   "   containers (batch embedding sweeps) - that split is the whole ops model."),
  ("2. THE PRIVACY ARGUMENT",
   "Regulated RAG: raw customer text must not leave the account. With SCS the chunker + embedding model run",
   "   inside; embeddings write STRAIGHT into Snowflake tables (Day 22 Q12 schema); queries embed inside the",
   "   service too. An external LLM API by contrast ships prompt text out of the account (PII in prompts,",
   "   per-token cost, egress review, latency). That trade is THE reason SCS exists."),
  ("3. RAG END-TO-END ON SCS",
   "(a) stage docs -> (b) SCS JOB: chunk + embed with open weights (e.g. bge/e5 on GPU) -> (c) embeddings table",
   "   + vector index (Snowflake vector type or a custom ANN service) -> (d) SCS SERVICE: query embedding +",
   "   hybrid retrieval + cross-encoder rerank at query time -> (e) optional Cortex LLM call only for the final",
   "   generation step (smallest possible egress, or fully on-prem weights in SCS for total lockdown).\n"
   "   Cortex functions vs open weights: managed = zero ops + per-function pricing but fixed models; open",
   "   weights = fine-tuning + IP control + GPU ops burden. Most regulated shops: open embedding/reranker",
   "   (SCS) + managed generation (Cortex) - egress only for the SUMMARY, not the corpus."),
  ("4. OPERATIONS REALITY",
   "Billing = compute-pool node-hours (GPU pools are expensive - size them to throughput, scale to zero for", "   jobs). Cold starts (image pull + model load: tens of seconds) matter for spiky traffic - keep a min=1 warm",
   "   instance on interactive endpoints. Pool starvation: GPU nodes are capacity-constrained per region - have a",
   "   CPU/fallback tier. Autoscaling is service-instance-level (not full k8s HPA with custom metrics) - for",
   "   complex serving topologies you may genuinely need real k8s, and saying so is a GOOD answer, not defeat."),
  ("5. DECISION TABLE",
   "Scenario -> pick:\n"
   "   - Standard NLP/LLM tasks, no fine-tune, cost-light -> Cortex managed functions (no infra at all)\n"
   "   - Custom/fine-tuned model or privacy lockdown on embeddings -> SCS (GPU pool, jobs for batch)\n"
   "   - Heavy multi-model platform with canary/serving meshes -> external GPU infra (Snowflake reads via",
   "     egress-controlled federation)\n"
   "   - Batch-only embedding of a corpus nightly -> SCS JOBS (scale to zero) beats a warm service by a mile."),
]
for topic, *lines in TALK_TRACK:
    print(f"{topic}\n{'-'*len(topic)}")
    for ln in lines:
        print(ln)
    print()
print("Q15 OK: pieces, privacy, RAG pipeline, ops, decision table")

## Q16 — Monitoring & Orchestration: Declarative Automation Bundles (DABs) — Promoting Data Pipelines Like Software [FAANG]

**Interviewer (Databricks, field engineering):** Your team's pipeline moves between dev → staging → prod by CLICKING: re-create jobs, re-point paths, re-grant. Every release has a checklist and every checklist has failed once. Make this a software release.

**Requirements — design the DAB-based CI/CD:**
1. **What a DAB is**: one `databricks.yml` + resources as code (jobs, pipelines, notebooks/files, dashboards, alerts, quality monitors) + per-environment targets that parameterize the SAME resources (catalog/schema names, cluster sizes, schedules). What does `bundle validate` / `bundle deploy` actually do (diff + apply against the workspace), and why does drift become visible instead of invisible?
2. **The promotion flow**: feature branch → PR (CI runs `validate` + unit tests on transformed data) → deploy to `dev` target → smoke job → deploy `staging` (integrated tests) → tag → deploy `prod`. Where do secrets come from (never in the YAML), and how do job definitions stay identical across envs with only a `targets:` override?
3. **What DABs unlock for monitoring**: alerts as resources ship WITH the pipeline (no orphaned alert after a job rename); `{{bundle.target}}` variables in job names/tags for cost attribution; run IDs queryable in `system.lakeflow.*` with the bundle identity — how does that close the loop back to reliability analytics?
4. **Limits & anti-patterns**: what NOT to put in a bundle (uploaded ad-hoc artifacts, prod data), the one-job-per-bundle smell, and when a separate orchestration layer (Airflow across systems) is still right vs Jobs-native orchestration within the lakehouse.

**What the interviewer wants**: infrastructure-as-code thinking applied to DATA, the CI/CD flow with tests, and knowing DABs don't replace cross-system orchestrators.

In [0]:
# Solution Q16 — DABs talk track: structure, promotion flow, monitoring loop, limits
DAB_YAML = """
# databricks.yml (sketch - the shape interviewers want to see)
bundle:
  name: payments_pipeline

resources:
  jobs:
    payments_daily:
      name: payments_daily_${bundle.target}
      tags: { team: payments, env: ${bundle.target} }        # cost attribution + filtering
      tasks:
        - task_key: ingest
          pipeline_task: { pipeline_id: ${resources.pipelines.payments.id} }
        - task_key: quality_gate
          depends_on: [{task_key: ingest}]
          sql_task: { file: ../sql/quality_gate.sql }
        - task_key: publish
          depends_on: [{task_key: quality_gate, outcome: ALL_SUCCESS }]  # Run If semantics as code
  pipelines:
    payments: { catalog: ${var.catalog}, target: silver_${bundle.target} }
  alerts:
    payments_freshness:                                        # alert ships WITH the job - no orphans
      query: {file: ../sql/freshness_check.sql}
      schedule: {quartz_cron_expression: '0 0 7 * * ?'}

targets:
  dev:    {variables: {catalog: dev_main}}      # same resources, different binding
  staging:{ variables: {catalog: staging_main}}
  prod:   { mode: production, variables: {catalog: main}, run_as: {service_principal_name: svc-payments} }
"""
print(DAB_YAML)

FLOW = [
  ("PR", "feature branch -> CI: databricks bundle validate + unit tests (transforms are pure functions)",
   "   + schema-diff check (data contract as a TEST, not a hope)"),
  ("deploy dev", "bundle deploy -t dev; smoke job runs on a scoped slice; failures post to the PR"),
  ("deploy staging", "same command, different target: integrated tests on masked prod-scale data;",
   "   the ONLY thing that changed is the targets binding - that is the whole point"),
  ("tag & prod", "git tag v1.4.2 -> bundle deploy -t prod (run_as service principal, never a human);",
   "   release notes generated from the bundle diff: validate shows added/changed resources BEFORE apply"),
]
print("Promotion flow:")
for step, *lines in FLOW:
    print(f"  {step:16s} {lines[0]}")
    for ln in lines[1:]:
        print(f"                   {ln.strip()}")
print()
print("Monitoring loop: alerts/dashboards are bundle RESOURCES -> they rename/move with the job;")
print("job names carry ${bundle.target} + team tag -> system.billing.usage and system.lakeflow.job* runs")
print("attribute cleanly back to the BUNDLE (which git SHA deployed what) - reliability analytics (MTTR,")
print("flakiness, Day 20) become per-release measurable. Drift is a deploy diff away, not an archaeology dig.")
print()
print("Limits: keep prod DATA out of bundles (resources only); one bundle = one deployable product")
print("(a lakehouse platform ends up with dozens, not one mega-bundle); DABs orchestrate WITHIN the")
print("lakehouse - Airflow/managed orchestration still owns cross-system flows (warehouse -> lake -> ML).")
print("Q16 OK: yaml shape, 4-step promotion, monitoring loop, honest limits")

## Q17 — Monitoring: Real-Time Anomaly Detection — Why Naive EWMA Misses Seasonal Data, and What to Do Instead [HARD]

**Interviewer (Uber, on-call platform):** Alerting on a 14-day hourly metric (orders per hour). A naive EWMA control chart (level + variance, z > 3) is proposed. You will find out it is broken — and fix it.

**Requirements**
1. **Build the series**: 24h sine seasonality (amplitude 8) + noise σ1.2 + a weekend dip (−20) + a real spike (+40, hour 243) and a real drop (−30, hour 116).
2. **Detector A — EWMA**: `ewma = α·x + (1−α)·ewma`, variance via the EWMA update, z = |x−ewma|/√ewvar, flag z>3. **Empirically show** the spike and drop are MISSED — explain the mechanism (the seasonal swing inflates ewvar to ~450 vs true noise 1.4, so ±40 events sit at z≈1.5).
3. **Detector B — seasonal decomposition + robust MAD**: hour-of-day × weekday/weekend median profile from a 7-day training window → residual → robust z via MAD (1.4826·MAD). Show it flags EXACTLY the 2 real anomalies with **zero** weekend false positives, and explain why MAD (not σ) keeps the contaminated training window from wrecking the bands (Days 18/21 used MAD on tables — this is the streaming-alerting application).
4. **Answer the follow-up**: when do you reach for EWMA anyway (no seasonality, drift detection)? What changes at 10k series (per-series profiles cost memory — sketch-based/quantile summaries, Day 18)? Where does STL/Prophet-style decomposition fit vs simple hour-of-week profiles?

**What the interviewer wants**: detectors evaluated against known ground truth, the honest failure mechanism (seasonality inflates variance → misses), and robust statistics instead of parametric σ.

In [0]:
# Solution Q17 — EWMA vs seasonality-aware detection (validated simulator)
import numpy as np

rng = np.random.default_rng(7)
hours = np.arange(14 * 24)
amp, noise, wkd = 8.0, 1.2, 20.0
vals = 100 + amp * np.sin(2 * np.pi * hours / 24) + rng.normal(0, noise, len(hours))
vals[10 * 24 + 3] += 40      # genuine spike (Wednesday)
vals[4 * 24 + 20] -= 30      # genuine drop (Wednesday, keeps the training window clean except this)
weekend = np.array([(h // 24) % 7 in (5, 6) for h in hours])
vals = vals - weekend * wkd

# --- Detector A: EWMA level + EWMA variance, z > 3
alpha = 0.3
ewma = np.zeros(len(vals)); ewvar = np.zeros(len(vals))
ewma[0], ewvar[0] = vals[0], noise**2
for i in range(1, len(vals)):
    ewma[i] = alpha * vals[i] + (1 - alpha) * ewma[i - 1]
    ewvar[i] = (1 - alpha) * (ewvar[i - 1] + alpha * (vals[i] - ewma[i - 1])**2)
z_ewma = np.abs(vals - ewma) / np.sqrt(ewvar)
naive_flags = z_ewma > 3.0

# --- Detector B: hour x weekday/weekend median profile + robust MAD bands
hod = hours % 24
key = np.array([f"{h}_{'w' if w else 'd'}" for h, w in zip(hod, weekend)])
train = hours < 7 * 24
med = {kk: np.median(vals[train & (key == kk)]) for kk in set(key[train])}
resid = vals - np.array([med[k] for k in key])
rmed = np.median(resid)
mad = np.median(np.abs(resid - rmed))
z_seas = np.abs(resid - rmed) / (1.4826 * mad)
seas_flags = z_seas > 6.0

s_ix, d_ix = 10 * 24 + 3, 4 * 24 + 20
print(f"EWMA:     z(spike)={z_ewma[s_ix]:.2f}  z(drop)={z_ewma[d_ix]:.2f}  flags={int(naive_flags.sum())}  (ewvar inflated to {ewvar.max():.0f} vs true noise {noise**2:.1f})")
print(f"Seasonal: z(spike)={z_seas[s_ix]:.2f}  z(drop)={z_seas[d_ix]:.2f}  flags={int(seas_flags.sum())}  weekend FPs={int((seas_flags & weekend).sum())}")
assert not naive_flags[s_ix] and not naive_flags[d_ix]      # naive MISSES both
assert seas_flags[s_ix] and seas_flags[d_ix]                 # seasonal catches both
assert (seas_flags & weekend).sum() == 0 and seas_flags.sum() == 2   # and nothing else
print("Q17 OK: mechanism proven - seasonal swing inflates EWMA variance ~300x, so real events sit at z=1.5;")
print("   the hour-of-week profile + MAD residual flags exactly the 2 true anomalies with 0 FPs.")
print()
print("Follow-ups: (a) EWMA is right when there is NO seasonality (infrastructure gauges, drift detection);")
print("(b) at 10k series store hour-of-week PROFILES (48 floats each) or t-digest quantile sketches (Day 18),")
print("    never raw samples; (c) full STL/Prophet decomposition earns its complexity only when the seasonal")
print("    pattern itself drifts (holiday calendars, growth) - start with hour-of-week medians, upgrade on evidence.")

## Q18 — Monitoring: PII Access Compliance Monitoring — Privacy Engineering as a Data Pipeline [HARD]

**Interviewer (Meta, privacy infra):** Auditors ask three questions your dashboards cannot answer: *who read PII without being in the authorized group?*, *whose access is stale enough to revoke?*, and *which PII tables have outlived their retention policy?* Build the compliance monitor as a data pipeline (Day 19 sketched audit-log scanning as one step — here it is the full design).

**Requirements**
1. **Inputs**: a simulated `system.access`-style audit log (ts, principal, action, object, has_pii flag, pii_reader group membership, last_read_days_ago) and a retention-policy table (object, policy_days, oldest_data_days).
2. **Three checks in ONE SQL pass**:
 - **F1 unauthorized PII access**: SELECT/GRANT on PII-tagged objects by principals NOT in `pii_readers` (flag the GRANT case separately — privilege escalation, not just reads);
 - **F2 stale access**: principals with no read in >90 days → revoke candidates (least privilege);
 - **F3 retention overdue**: tables whose oldest data exceeds their policy → purge/legal-review queue.
3. **Operationalize**: PII detection (governed tags / column classification feeding has_pii), scheduling (daily SQL alert — Day 14 metrics-table pattern), severity routing (F1 page security, F2 weekly digest, F3 legal), and the failure mode of tag-based detection (untagged PII is invisible — monitor tag coverage itself).
4. **Assert the verdicts** on the sample: bob and mallory trip F1 (alice does NOT — she is authorized), frank trips F2, and `main.sec.raw_emails` trips F3 (120d data vs 90d policy).

**What the interviewer wants**: compliance as CODE over audit logs, least-privilege + retention as monitorable metrics, and the humility that tags/classification are the weak link.

In [0]:
# Solution Q18 — PII access compliance monitor (one SQL pass, three checks)
from pyspark.sql import functions as F

audit = spark.createDataFrame(
    [("2026-11-04 09:12","alice@x.com","SELECT","main.gold.customer_pii",True,True,0),
     ("2026-11-04 09:15","bob@x.com","SELECT","main.gold.customer_pii",True,False,0),
     ("2026-11-04 09:20","svc-bi@x.com","SELECT","main.gold.orders",False,True,2),
     ("2026-11-04 09:30","eve@x.com","SELECT","main.gold.customer_pii",True,True,0),
     ("2026-11-04 09:41","mallory@x.com","GRANT","main.gold.customer_pii",True,False,0),
     ("2026-11-04 10:05","alice@x.com","SELECT","main.sec.raw_emails",True,True,0),
     ("2026-11-04 10:20","frank@x.com","SELECT","main.gold.orders",False,False,120),
     ("2026-11-04 10:30","bob@x.com","SELECT","main.sec.raw_emails",True,False,0)],
    "ts string, principal string, action string, object string, has_pii boolean, pii_reader boolean, last_read_days_ago int")
audit.createOrReplaceTempView("audit")
retention = spark.createDataFrame(
    [("main.sec.raw_emails", 90, 120), ("main.gold.customer_pii", 400, 380), ("main.gold.orders", 400, 35)],
    "object string, policy_days int, oldest_data_days int")
retention.createOrReplaceTempView("retention")

report = spark.sql("""
  WITH pii_access AS (
    SELECT principal, action, object, ts,
           CASE WHEN has_pii AND NOT pii_reader AND action IN ('SELECT','GRANT')
                THEN 'F1: PII access without pii_readers membership' END AS violation
    FROM audit WHERE has_pii
  )
  SELECT 'F1_unauthorized_pii_access' AS check, principal, object, ts, violation AS detail
  FROM pii_access WHERE violation IS NOT NULL
  UNION ALL
  SELECT 'F1_grant_pii_outsider' AS check, principal, object, ts,
         'F1: GRANT on PII object by principal outside pii_readers' AS violation
  FROM audit WHERE action = 'GRANT' AND has_pii AND NOT pii_reader
  UNION ALL
  SELECT 'F2_stale_access' AS check, principal, CAST(NULL AS STRING) AS object, CAST(NULL AS STRING) AS ts,
         concat('F2: no read in ', CAST(mx AS STRING), 'd (>90d) - revoke candidate') AS violation
  FROM (SELECT principal, max(last_read_days_ago) AS mx
        FROM audit WHERE principal LIKE '%@x.com' GROUP BY principal)
  WHERE mx > 90
  UNION ALL
  SELECT 'F3_retention_overdue' AS check, CAST(NULL AS STRING), r.object, CAST(NULL AS STRING),
         concat('F3: oldest data ', CAST(oldest_data_days AS STRING),
                'd exceeds policy ', CAST(policy_days AS STRING), 'd - purge / legal review') AS violation
  FROM retention r WHERE r.oldest_data_days > r.policy_days
""")
report.orderBy("check", "principal").show(truncate=False)

rows = {(r["check"], r["principal"]) for r in report.collect()}
assert ("F1_unauthorized_pii_access", "bob@x.com") in rows
assert ("F1_unauthorized_pii_access", "mallory@x.com") in rows
assert ("F1_unauthorized_pii_access", "alice@x.com") not in rows      # authorized reader: no violation
assert ("F2_stale_access", "frank@x.com") in rows
assert ("F2_stale_access", "alice@x.com") not in rows
f3 = [r for r in report.collect() if r["check"] == "F3_retention_overdue"]
assert len(f3) == 1 and f3[0]["object"] == "main.sec.raw_emails"
print("Q18 OK: bob + mallory trip F1 (read + GRANT escalation), frank trips F2 (120d stale), raw_emails trips F3")
print()
print("Operationalization: has_pii comes from governed tags / column classification (monitor TAG COVERAGE too -")
print("untagged PII is invisible to this monitor); run daily as a SQL alert writing to a metrics table (Day 14)");
print("severity: F1 -> page security SAME DAY (active exfiltration risk), F2 -> weekly access-review digest,")
print("F3 -> legal ticket with the purge candidate list. In prod read system.access + system.query.audit.")

# 🏁 Day 22 Scorecard — FAANG / Top Product Company Level (Week 4, Day 1)

Score each question 0–3: **0** = could not start · **1** = needed the solution · **2** = got most of it with hints · **3** = interview-ready.

| Q | Category | Topic | Tag | Score (0–3) |
| --- | --- | --- | --- | --- |
| Q1 | SQL Advanced | OHLC candles + as-of quote enrichment | FAANG | |
| Q2 | SQL Advanced | A/B integrity: SRM chi-square + CUPED | FAANG | |
| Q3 | Python | Count-Min Sketch heavy hitters | FAANG | |
| Q4 | Python | Text-to-SQL guardrails | FAANG | |
| Q5 | PySpark | Streaming dedup state semantics | HARD | |
| Q6 | PySpark | IVF vector index + ANN recall trade-off | FAANG | |
| Q7 | Spark Opt | Shuffle mechanics + push-based shuffle | HARD | |
| Q8 | Spark Opt | Lakehouse Federation pushdown | HARD | |
| Q9 | ETL | GenAI doc ingestion: idempotent + DLQ + circuit breaker | FAANG | |
| Q10 | ETL | Cross-cloud DR architecture | HARD | |
| Q11 | ETL | Streaming Tables vs Materialized Views | HARD | |
| Q12 | Data Modeling | RAG corpus as governed data product | FAANG | |
| Q13 | Data Modeling | Time-series metrics platform model | HARD | |
| Q14 | Snowflake | Zero-trust security stack | HARD | |
| Q15 | Snowflake | Snowpark Container Services | HARD | |
| Q16 | Monitoring | DABs env promotion + CI/CD | FAANG | |
| Q17 | Monitoring | EWMA pitfalls + seasonal MAD detection | HARD | |
| Q18 | Monitoring | PII access compliance monitor | HARD | |

**Totals**: 18 questions · 8 categories · 10 [HARD] + 8 [FAANG]

*Interpretation*: 45+ = FAANG-loop ready on data engineering. 36–44 = strong; drill the sub-3 rows. <36 = revisit the Week 3 notebooks (Days 15–21) for the foundations these questions build on.

**New concepts locked in today**: struct-trick OHLC + as-of joins · SRM gate + CUPED identity · count-min frequency sketches · LLM-SQL guardrails · dedup state eviction rules · IVF probe/recall · shuffle write/read/Magnet · federation pushdown verification · content-hash idempotency + retry budgets · cross-cloud DR tiers · ST vs MV refresh semantics · RAG corpus lineage · metrics-platform cardinality · zero-trust perimeter → keys → identity → proof · GPU serving in-account · DABs promotion · seasonal MAD alerting · privacy compliance as code.

**Tomorrow (Day 23)**: Week 4 continues — FAANG/HARD. Fresh candidates: streaming joins across watermarks (beyond Day 19), Flink vs Spark windowing internals, GenAI eval pipelines (LLM-as-judge), sketch-based quantiles at scale (t-digest/KLL), capacity burn-down forecasting, data product SLA contracts, multi-tenant catalog isolation at platform scale, Iceberg vs Delta deep internals, Kubernetes-based Spark, cost attribution per workload.